In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("marketing_campaign_data_dirty.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (52500, 16)

Columns:
['customer_id', 'date', 'channel', 'campaign', 'age', 'gender', 'region', 'device', 'is_new_customer', 'prior_purchases', 'pages_viewed', 'session_duration_sec', 'email_opened', 'acquisition_cost', 'converted', 'revenue']

Data types:
customer_id                 str
date                        str
channel                     str
campaign                    str
age                     float64
gender                      str
region                      str
device                      str
is_new_customer           int64
prior_purchases           int64
pages_viewed            float64
session_duration_sec    float64
email_opened            float64
acquisition_cost            str
converted               float64
revenue                     str
dtype: object

First 5 rows:
  customer_id        date         channel          campaign   age  gender  \
0         NaN  2025-07-25  Organic Search       SEO Content  42.0    Male   
1     C149247  2024-05-27           email

In [11]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df.columns.tolist())

['customer_id', 'date', 'channel', 'campaign', 'age', 'gender', 'region', 'device', 'is_new_customer', 'prior_purchases', 'pages_viewed', 'session_duration_sec', 'email_opened', 'acquisition_cost', 'converted', 'revenue']


In [13]:
text_columns = df.select_dtypes(include="str").columns

In [14]:
missing_values = [
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "None",
    "none",
    "-"
]

df = df.replace(missing_values, np.nan)

text_columns = df.select_dtypes(include="str").columns

for col in text_columns:
    df[col] = df[col].str.strip()

In [15]:
print(df.isna().sum())

customer_id              539
date                     122
channel                    0
campaign                1036
age                     3122
gender                  1983
region                  2881
device                  1502
is_new_customer            0
prior_purchases            0
pages_viewed            2060
session_duration_sec    2605
email_opened            2115
acquisition_cost           0
converted                506
revenue                 1502
dtype: int64


In [16]:
# Fix impossible ages first
df.loc[
    (df["age"] < 18) | (df["age"] > 100),
    "age"
] = np.nan

# Then fill missing age
df["age"] = df["age"].fillna(df["age"].median())

In [17]:
categorical_fill = [
    "campaign",
    "gender",
    "region",
    "device"
]

for col in categorical_fill:
    df[col] = df[col].fillna("Unknown")

In [18]:
df["pages_viewed"] = df["pages_viewed"].fillna(
    df["pages_viewed"].median()
)

df["session_duration_sec"] = df["session_duration_sec"].fillna(
    df["session_duration_sec"].median()
)

In [19]:
df = df.dropna(subset=["customer_id", "date"])

In [20]:
print(df.isna().sum())

customer_id                0
date                       0
channel                    0
campaign                   0
age                        0
gender                     0
region                     0
device                     0
is_new_customer            0
prior_purchases            0
pages_viewed               0
session_duration_sec       0
email_opened            2092
acquisition_cost           0
converted                500
revenue                 1480
dtype: int64


In [21]:
print(df["email_opened"].value_counts(dropna=False))

email_opened
0.0    27384
1.0    22365
NaN     2092
Name: count, dtype: int64


In [22]:
df["email_opened"] = df["email_opened"].fillna(0).astype(int)

In [23]:
print(df["converted"].value_counts(dropna=False))

converted
 0.0    44461
 1.0     6380
 NaN      500
 2.0      273
-1.0      227
Name: count, dtype: int64


In [24]:
print(
    df[df["converted"].isna()]["revenue"]
    .describe()
)

count     483
unique     41
top       0.0
freq      384
Name: revenue, dtype: object


In [25]:
print(
    df[
        df["converted"].isna() &
        df["revenue"].notna()
    ][["converted", "revenue"]].head(20)
)

      converted revenue
57          NaN     0,0
64          NaN     0.0
106         NaN     0.0
194         NaN    $0.0
259         NaN   165.0
368         NaN     0.0
739         NaN  $44.37
789         NaN     0.0
992         NaN     0.0
1051        NaN     0,0
1101        NaN     0.0
1202        NaN     0.0
1264        NaN     0.0
1330        NaN     0.0
1423        NaN    $0.0
1435        NaN     0.0
1578        NaN     0.0
1648        NaN     0,0
1712        NaN     0.0
1971        NaN   61.62


In [26]:
df["converted"] = df["converted"].astype("Int64")

In [28]:
print(
    df.groupby("converted", dropna=False)["revenue"]
      .agg(["count", "mean", "median", "min", "max"])
)

TypeError: dtype 'str' does not support operation 'mean'

In [29]:
# Convert revenue to numeric
df["revenue"] = (
    df["revenue"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["revenue"] = pd.to_numeric(
    df["revenue"],
    errors="coerce"
)

print(df["revenue"].dtype)

Float64


In [30]:
print(
    df.groupby("converted", dropna=False)["revenue"]
      .agg(["count", "mean", "median", "min", "max"])
)

           count        mean  median   min      max
converted                                          
-1           224       10.23     0.0 -40.0   291.57
0          43176   66.455508     0.0 -40.0  25000.0
1           6221  344.694943    66.5 -40.0  61388.0
2            257    4.164008     0.0   0.0   122.68
<NA>         483    6.908737     0.0   0.0   229.55


In [31]:
df["converted"] = pd.to_numeric(
    df["converted"],
    errors="coerce"
)

print(df["converted"].value_counts(dropna=False))

converted
0       44461
1        6380
<NA>      500
2         273
-1        227
Name: count, dtype: Int64


In [32]:
invalid_converted = ~df["converted"].isin([0, 1])

print("Invalid converted values:")
print(df.loc[invalid_converted, "converted"].value_counts(dropna=False))

Invalid converted values:
converted
<NA>    500
2       273
-1      227
Name: count, dtype: Int64


In [33]:
df.loc[
    ~df["converted"].isin([0, 1]),
    "converted"
] = np.nan

df["converted"] = df["converted"].astype("Int64")

In [34]:
print(
    df.groupby("converted", dropna=False)["revenue"]
      .agg(["count", "mean", "median", "min", "max"])
)

           count        mean  median   min      max
converted                                          
0          43176   66.455508     0.0 -40.0  25000.0
1           6221  344.694943    66.5 -40.0  61388.0
<NA>         964    6.948745     0.0 -40.0   291.57


In [35]:
print(df.isna().sum())

customer_id                0
date                       0
channel                    0
campaign                   0
age                        0
gender                     0
region                     0
device                     0
is_new_customer            0
prior_purchases            0
pages_viewed               0
session_duration_sec       0
email_opened               0
acquisition_cost           0
converted               1000
revenue                 1480
dtype: int64


In [36]:
df["customer_id"] = (
    df["customer_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [37]:
print(df["customer_id"].head(20))
print("\nMissing customer IDs:", df["customer_id"].isna().sum())

1     C149247
2     C105848
3     C128416
4     C123960
5     C107599
6     C102711
7     C118119
8     C133435
9     C117124
10    C140863
11    C113369
12    C132934
13    C149596
14    C133337
15    C112296
16    C132456
17    C107425
18    C119387
19    C123062
20    C138359
Name: customer_id, dtype: string

Missing customer IDs: 0


In [38]:
valid_id = df["customer_id"].str.match(r"^C\d+$", na=False)

print("Invalid customer IDs:", (~valid_id & df["customer_id"].notna()).sum())

Invalid customer IDs: 0


In [39]:
df = df.dropna(subset=["customer_id"])

In [40]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Exact duplicates removed:", before - after)
print("New shape:", df.shape)

Exact duplicates removed: 1982
New shape: (49859, 16)


In [41]:
duplicate_key = [
    "customer_id",
    "date",
    "campaign"
]

near_duplicates = df[
    df.duplicated(subset=duplicate_key, keep=False)
].sort_values(duplicate_key)

print(near_duplicates)

      customer_id        date         channel          campaign   age  gender  \
14280     C100033  2024-07-30           Email     Cart Recovery  18.0  Female   
42563     C100033  2024-07-30           Email     Cart Recovery  18.0  Female   
643       C100146  2024-11-22           Email    Welcome Series  29.0  Female   
14530     C100146  2024-11-22           Email    Welcome Series  29.0  Female   
18568     C100176  2024-09-11        Referral  Friends & Family  18.0  Female   
...           ...         ...             ...               ...   ...     ...   
52335     C149844  2024-11-03    Social Media     Summer Launch  39.0  Female   
2369      C149860  2025-10-29  Organic Search       SEO Content  29.0    Male   
39422     C149860  2025-10-29  Organic Search       SEO Content  29.0    Male   
9148      C149878  2025-06-11    Social Media     Summer Launch  51.0       F   
49488     C149878  2025-06-11    Social Media     Summer Launch  51.0       F   

              region  devic

In [42]:
print(
    near_duplicates.groupby(duplicate_key).size()
    .sort_values(ascending=False)
    .head(20)
)

customer_id  date          campaign        
C100033      2024-07-30    Cart Recovery       2
C100146      2024-11-22    Welcome Series      2
C100176      2024-09-11    Friends & Family    2
C100188      2024-12-08    Summer Launch       2
C100286      2024-03-21    Influencer Push     2
C100404      22/06/2024    Cart Recovery       2
C100505      2024-06-03    Generic Terms       2
C100573      2024-03-20    SEO Content         2
C100622      2024-12-26    SEO Content         2
C100712      01/02/2025    Generic Terms       2
C100729      2024-07-28    Influencer Push     2
C100771      Dec 03, 2025  SEO Content         2
C100795      Dec 17, 2025  Generic Terms       2
C100912      2024-11-17    Friends & Family    2
C100957      09/11/2024    SEO Content         2
C100976      2024-08-13    Summer Launch       2
C101007      07/08/2025    Cart Recovery       2
C101099      2024-08-17    Newsletter          2
C101157      2025-12-29    SEO Content         2
C101174      2025-11-11  

In [43]:
categorical_columns = [
    "channel",
    "gender",
    "region",
    "device"
]

In [44]:
for col in categorical_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

In [45]:
for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- channel ---
channel
social media      10826
email              9627
paid search        9451
organic search     8954
display ads        5899
referral           3931
organic             161
socal media         134
social_media        127
social-media        121
emial               119
display             116
e-mail              112
paid serch          110
paidsearch          106
refferal             65
Name: count, dtype: int64[pyarrow]

--- gender ---
gender
female        23001
male          21940
unknown        1883
non-binary     1327
m               442
f               428
non binary      425
other           413
Name: count, dtype: int64[pyarrow]

--- region ---
region
north america    18052
europe           12925
asia-pacific      9839
latin america     4398
unknown           2738
apac               402
n. america         398
eu                 397
asia pacific       373
latam              337
Name: count, dtype: int64[pyarrow]

--- device ---
device
mobile     27084
desktop   

In [46]:
channel_mapping = {
    "email": "Email",
    "e-mail": "Email",
    "mail": "Email",

    "social media": "Social Media",
    "socialmedia": "Social Media",
    "social": "Social Media",
    "socal media": "Social Media",
    "social med": "Social Media",

    "organic search": "Organic Search",
    "organic": "Organic Search",
    "seo": "Organic Search",

    "paid search": "Paid Search",
    "paid": "Paid Search",
    "sem": "Paid Search",

    "display ads": "Display Ads",
    "display ad": "Display Ads",
    "display": "Display Ads",

    "referral": "Referral",
    "referrals": "Referral"
}

df["channel"] = df["channel"].map(channel_mapping)

In [47]:
print(df["channel"].value_counts(dropna=False))

channel
Social Media      10960
Email              9739
Paid Search        9451
Organic Search     9115
Display Ads        6015
Referral           3931
NaN                 648
Name: count, dtype: int64


In [48]:
gender_mapping = {
    "male": "Male",
    "m": "Male",
    "man": "Male",

    "female": "Female",
    "f": "Female",
    "woman": "Female",

    "other": "Other",
    "o": "Other"
}

df["gender"] = df["gender"].map(gender_mapping)

print(df["gender"].value_counts(dropna=False))

gender
Female    23429
Male      22382
NaN        3635
Other       413
Name: count, dtype: int64


In [49]:
df["gender"] = df["gender"].fillna("Unknown")

print(df["gender"].value_counts(dropna=False))

gender
Female     23429
Male       22382
Unknown     3635
Other        413
Name: count, dtype: int64


In [50]:
df["gender"].fillna(df["gender"].mode()[0])

1          Male
2          Male
3        Female
4        Female
5        Female
          ...  
52494      Male
52495    Female
52496    Female
52498      Male
52499    Female
Name: gender, Length: 49859, dtype: str

In [51]:
print("Missing gender:", df["gender"].isna().sum())

Missing gender: 0


In [52]:
print(df["region"].value_counts(dropna=False))

region
north america    18052
europe           12925
asia-pacific      9839
latin america     4398
unknown           2738
apac               402
n. america         398
eu                 397
asia pacific       373
latam              337
Name: count, dtype: int64[pyarrow]


In [53]:
region_mapping = {
    "north": "North",
    "n": "North",

    "south": "South",
    "s": "South",

    "east": "East",
    "e": "East",

    "west": "West",
    "w": "West"
}

df["region"] = df["region"].map(region_mapping)

In [54]:
device_mapping = {
    "mobile": "Mobile",
    "phone": "Mobile",
    "smartphone": "Mobile",

    "desktop": "Desktop",
    "pc": "Desktop",
    "computer": "Desktop",

    "tablet": "Tablet",
    "tab": "Tablet"
}

df["device"] = df["device"].map(device_mapping)

print(df["device"].value_counts(dropna=False))

device
Mobile     27610
Desktop    17109
Tablet      3721
NaN         1419
Name: count, dtype: int64


In [55]:
print(df["date"].head(20))

1       2024-05-27
2       2025-03-23
3       2025-03-25
4       2024-10-01
5       2024-08-16
6     Oct 02, 2025
7       2024-09-05
8       2024-07-18
9       2025-10-07
10      2024-06-25
11      21/01/2025
12      2024-07-03
13    Dec 30, 2024
14      2024-07-01
15      2024-11-07
16      11/12/2025
17      2025-07-15
18      2025-05-03
19      2025-12-26
20      2025-02-16
Name: date, dtype: str


In [56]:
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce",
    dayfirst=False
)

In [57]:
print("Invalid dates:", df["date"].isna().sum())

Invalid dates: 12049


In [58]:
df["date"] = df["date"].replace(
    ["", "N/A", "NA", "na", "null", "NULL", "None"],
    np.nan
)

In [59]:
df["date"] = pd.to_datetime(
    df["date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [60]:
print("Invalid dates:", df["date"].isna().sum())

Invalid dates: 12049


In [61]:
df = pd.read_csv("marketing_campaign_data_dirty.csv")

df["date"] = df["date"].astype("string").str.strip()

# Standardize obvious missing values
df["date"] = df["date"].replace(
    ["", "N/A", "NA", "na", "null", "NULL", "None"],
    np.nan
)

# Start with missing dates
parsed_date = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

# ISO format: YYYY-MM-DD
mask_iso = df["date"].str.match(
    r"^\d{4}-\d{2}-\d{2}$",
    na=False
)

parsed_date.loc[mask_iso] = pd.to_datetime(
    df.loc[mask_iso, "date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# DD/MM/YYYY
mask_dmy = df["date"].str.match(
    r"^\d{2}/\d{2}/\d{4}$",
    na=False
)

parsed_date.loc[mask_dmy] = pd.to_datetime(
    df.loc[mask_dmy, "date"],
    format="%d/%m/%Y",
    errors="coerce"
)

# Example: Mar 05, 2024
mask_text = df["date"].str.match(
    r"^[A-Za-z]{3} \d{2}, \d{4}$",
    na=False
)

parsed_date.loc[mask_text] = pd.to_datetime(
    df.loc[mask_text, "date"],
    format="%b %d, %Y",
    errors="coerce"
)

df["date"] = parsed_date

In [62]:
print("Invalid dates:", df["date"].isna().sum())

Invalid dates: 531


In [63]:
print(df[df["date"].isna()].head(20))

     customer_id date         channel          campaign   age      gender  \
49       C107412  NaT     Paid Search  Competitor Terms  18.0        Male   
105      C113665  NaT     Paid Search  Competitor Terms  45.0        Male   
178      C144768  NaT     PAID SEARCH  Competitor Terms  33.0      Female   
359      C106042  NaT        Referral  Friends & Family  18.0        Male   
396      C140959  NaT    Paid Search   Competitor Terms  27.0  Non-binary   
482      C109933  NaT           Email               NaN  42.0        Male   
534      C122119  NaT        Referral  Friends & Family  18.0      Female   
562      c123868  NaT  Organic Search       SEO Content  34.0        Male   
724      C142815  NaT  Organic Search       SEO Content  60.0        Male   
854      C108567  NaT    Social Media   Influencer Push  39.0        Male   
1051     C137924  NaT  Organic Search       SEO Content  46.0      Female   
1139     C134160  NaT  Organic Search       SEO Content  32.0      Female   

In [64]:
before = len(df)

df = df.dropna(subset=["date"]).copy()

after = len(df)

print("Rows removed:", before - after)
print("Remaining rows:", len(df))
print("Invalid dates:", df["date"].isna().sum())

Rows removed: 531
Remaining rows: 51969
Invalid dates: 0


In [65]:
print("Minimum date:", df["date"].min())
print("Maximum date:", df["date"].max())

Minimum date: 2024-01-01 00:00:00
Maximum date: 2025-12-30 00:00:00


In [66]:
print(df["date"].describe())

count                            51969
mean     2024-12-29 09:54:14.609478912
min                2024-01-01 00:00:00
25%                2024-06-28 00:00:00
50%                2024-12-30 00:00:00
75%                2025-06-28 00:00:00
max                2025-12-30 00:00:00
Name: date, dtype: object


In [67]:
numeric_columns = [
    "age",
    "sessions",
    "session_duration",
    "revenue",
    "cost",
    "converted"
]

In [68]:
df["revenue"] = (
    df["revenue"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["revenue"] = pd.to_numeric(
    df["revenue"],
    errors="coerce"
)

In [71]:
invalid_age = (
    (df["age"] < 18) |
    (df["age"] > 100)
)

print("Invalid ages:", invalid_age.sum())

Invalid ages: 519


In [72]:
df.loc[invalid_age, "age"] = np.nan

In [74]:
invalid_conversion = ~df["converted"].isin([0, 1])

print("Invalid conversion values:", invalid_conversion.sum())

df.loc[invalid_conversion, "converted"] = np.nan

Invalid conversion values: 996


In [75]:
df["converted"] = df["converted"].astype("Int64")

In [80]:
# Clean acquisition_cost

df["acquisition_cost"] = (
    df["acquisition_cost"]
    .astype("string")
    .str.replace("USD", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)

df["acquisition_cost"] = pd.to_numeric(
    df["acquisition_cost"],
    errors="coerce"
)

# Remove impossible negative values

df.loc[df["acquisition_cost"] < 0, "acquisition_cost"] = np.nan

In [81]:
print(df["acquisition_cost"].dtype)
print(df["acquisition_cost"].isna().sum())
print(df["acquisition_cost"].describe())

Float64
58
count       51911.0
mean       8.800015
std      177.584193
min             0.0
25%            0.36
50%            1.39
75%            2.21
max          5000.0
Name: acquisition_cost, dtype: Float64


In [82]:
print(df["revenue"].dtype)

Float64


In [83]:
df.loc[df["revenue"] < 0, "revenue"] = np.nan

In [84]:
print(df[["revenue", "acquisition_cost"]].dtypes)

revenue             Float64
acquisition_cost    Float64
dtype: object


In [86]:
df.loc[df["revenue"] < 0, "revenue"] = np.nan
df.loc[df["acquisition_cost"] < 0, "acquisition_cost"] = np.nan

In [87]:
print(df["revenue"].describe())

count        50374.0
mean      100.223173
std      1243.992047
min              0.0
25%              0.0
50%              0.0
75%              0.0
max          61388.0
Name: revenue, dtype: Float64


In [88]:
Q1 = df["revenue"].quantile(0.25)
Q3 = df["revenue"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

revenue_outliers = df[
    (df["revenue"] < lower_bound) |
    (df["revenue"] > upper_bound)
]

print("Potential revenue outliers:", len(revenue_outliers))

Potential revenue outliers: 6456


In [89]:
df.loc[
    (df["converted"] == 1) &
    (df["revenue"] <= 0),
    "converted"
] = np.nan

In [90]:
df.loc[
    (df["converted"] == 0) &
    (df["revenue"] > 0),
    "revenue"
] = np.nan

In [92]:
duplicate_key = [
    "customer_id",
    "date",
    "campaign"
]

near_duplicates = df[
    df.duplicated(
        subset=duplicate_key,
        keep=False
    )
].sort_values(duplicate_key)

print(near_duplicates)

      customer_id       date         channel          campaign   age  gender  \
1060      C100010 2025-12-26    Social Media       Retargeting   NaN  Female   
44685     C100010 2025-12-26    Social Media       Retargeting   NaN  Female   
14280     C100033 2024-07-30           Email     Cart Recovery  18.0  Female   
42563     C100033 2024-07-30           Email     Cart Recovery  18.0  Female   
38301     C100044 2024-06-29     DISPLAY ADS  Banner Awareness  34.0  Female   
...           ...        ...             ...               ...   ...     ...   
46448         NaN 2025-11-15  Organic Search       SEO Content   NaN  Female   
2618          NaN 2025-11-18           Email        Newsletter  26.0    Male   
46544         NaN 2025-11-18           Email        Newsletter  26.0    Male   
37385         NaN 2025-11-23  ORGANIC SEARCH       SEO Content  38.0    Male   
44748         NaN 2025-11-23  ORGANIC SEARCH       SEO Content  38.0    Male   

              region   device  is_new_c

In [93]:
df = df.drop_duplicates()

In [94]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

print(
    missing_summary
    .sort_values("missing_percent", ascending=False)
)

                      missing_count  missing_percent
age                            3470         6.942916
region                         2736         5.474299
session_duration_sec           2487         4.976090
revenue                        2456         4.914064
email_opened                   1998         3.997679
pages_viewed                   1956         3.913644
gender                         1888         3.777587
converted                      1708         3.417435
device                         1428         2.857200
campaign                        998         1.996839
customer_id                     499         0.998419
acquisition_cost                 57         0.114048
date                              0         0.000000
channel                           0         0.000000
is_new_customer                   0         0.000000
prior_purchases                   0         0.000000


In [95]:
df["age"] = df["age"].fillna(df["age"].median())

In [96]:
df["gender"] = df["gender"].fillna("Unknown")
df["region"] = df["region"].fillna("Unknown")
df["device"] = df["device"].fillna("Unknown")
df["channel"] = df["channel"].fillna("Unknown")

In [97]:
# Do not fabricate customer IDs.
df = df.dropna(subset=["customer_id"])

In [98]:
print(df.dtypes)

customer_id                        str
date                    datetime64[ns]
channel                            str
campaign                           str
age                            float64
gender                             str
region                             str
device                             str
is_new_customer                  int64
prior_purchases                  int64
pages_viewed                   float64
session_duration_sec           float64
email_opened                   float64
acquisition_cost               Float64
converted                        Int64
revenue                        Float64
dtype: object


In [99]:
print("Missing values:")
print(df.isna().sum())

Missing values:
customer_id                0
date                       0
channel                    0
campaign                 986
age                        0
gender                     0
region                     0
device                     0
is_new_customer            0
prior_purchases            0
pages_viewed            1930
session_duration_sec    2457
email_opened            1982
acquisition_cost          56
converted               1694
revenue                 2426
dtype: int64


In [100]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [101]:
print(
    "Invalid ages:",
    ((df["age"] < 18) | (df["age"] > 100)).sum()
)

Invalid ages: 0


In [102]:
print(
    "Invalid converted values:",
    (~df["converted"].isin([0, 1])).sum()
)

Invalid converted values: 1694


In [103]:
# Convert converted to numeric
df["converted"] = pd.to_numeric(
    df["converted"],
    errors="coerce"
)

# Count invalid values
invalid_converted = ~df["converted"].isin([0, 1]) & df["converted"].notna()

print("Invalid converted values:", invalid_converted.sum())

# Replace invalid values with NaN
df.loc[invalid_converted, "converted"] = np.nan

# Use nullable integer type
df["converted"] = df["converted"].astype("Int64")

# Verify
print(df["converted"].value_counts(dropna=False))

Invalid converted values: 0
converted
0       42441
1        5345
<NA>     1694
Name: count, dtype: Int64


In [104]:
# Check revenue by conversion status

print(
    df.groupby("converted", dropna=False)["revenue"]
      .agg(["count", "mean", "median", "min", "max"])
)

           count        mean  median   min      max
converted                                          
0          40210         0.0     0.0   0.0      0.0
1           5186  392.069117   74.86  10.0  61388.0
<NA>        1658     4.00968     0.0   0.0   291.57


In [105]:
print("Converted = 0 but revenue > 0:",
      ((df["converted"] == 0) & (df["revenue"] > 0)).sum())

print("Converted = 1 but revenue = 0:",
      ((df["converted"] == 1) & (df["revenue"] == 0)).sum())

print("Converted = 1 but revenue missing:",
      ((df["converted"] == 1) & (df["revenue"].isna())).sum())

print("Converted missing but revenue > 0:",
      (df["converted"].isna() & (df["revenue"] > 0)).sum())

Converted = 0 but revenue > 0: 0
Converted = 1 but revenue = 0: 0
Converted = 1 but revenue missing: 159
Converted missing but revenue > 0: 73


In [106]:
print(
    "Invalid converted values:",
    (~df["converted"].isin([0, 1])).sum()
)

Invalid converted values: 1694


In [107]:
print(
    "Negative revenue:",
    (df["revenue"] < 0).sum()
)

Negative revenue: 0


In [109]:
print(
    "Negative cost:",
    (df["acquisition_cost"] < 0).sum()
)

Negative cost: 0


In [110]:
print(
    "Invalid dates:",
    df["date"].isna().sum()
)

Invalid dates: 0


In [111]:
print("=" * 50)
print("FINAL DATASET AUDIT")
print("=" * 50)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())
print("Missing cells:", df.isna().sum().sum())

print("\nMissing values by column:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

print("\nChannel values:")
print(df["channel"].value_counts(dropna=False))

print("\nGender values:")
print(df["gender"].value_counts(dropna=False))

print("\nRegion values:")
print(df["region"].value_counts(dropna=False))

print("\nDevice values:")
print(df["device"].value_counts(dropna=False))

FINAL DATASET AUDIT
Rows: 49480
Columns: 16
Duplicate rows: 0
Missing cells: 11531

Missing values by column:
customer_id                0
date                       0
channel                    0
campaign                 986
age                        0
gender                     0
region                     0
device                     0
is_new_customer            0
prior_purchases            0
pages_viewed            1930
session_duration_sec    2457
email_opened            1982
acquisition_cost          56
converted               1694
revenue                 2426
dtype: int64

Data types:
customer_id                        str
date                    datetime64[ns]
channel                            str
campaign                           str
age                            float64
gender                             str
region                             str
device                             str
is_new_customer                  int64
prior_purchases                  int64
pages_view

In [112]:
channel_map = {
    "social media": "Social Media",
    "socal media": "Social Media",
    "social_media": "Social Media",
    "social-media": "Social Media",

    "email": "Email",
    "e-mail": "Email",
    "emial": "Email",

    "paid search": "Paid Search",
    "paidsearch": "Paid Search",
    "paid serch": "Paid Search",

    "organic search": "Organic Search",
    "organic": "Organic Search",

    "display ads": "Display Ads",
    "display": "Display Ads",

    "referral": "Referral",
    "refferal": "Referral"
}

df["channel"] = (
    df["channel"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(channel_map)
)

print(df["channel"].value_counts(dropna=False))

channel
Social Media      11109
Email              9781
Paid Search        9590
Organic Search     9059
Display Ads        5975
Referral           3966
Name: count, dtype: int64[pyarrow]


In [113]:
gender_map = {
    "female": "Female",
    "f": "Female",

    "male": "Male",
    "m": "Male",

    "non binary": "Non-binary",
    "non-binary": "Non-binary",

    "other": "Other",
    "unknown": "Unknown"
}

df["gender"] = (
    df["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(gender_map)
)

print(df["gender"].value_counts(dropna=False))

gender
Female        23248
Male          22219
Unknown        1870
Non-binary     1732
Other           411
Name: count, dtype: int64[pyarrow]


In [114]:
print(df["region"].value_counts(dropna=False))

region
North America    17563
Europe           12470
Asia-Pacific      9753
Latin America     4369
Unknown           2712
APAC               401
N. America         396
EU                 395
Asia Pacific       370
north america      362
europe             357
LATAM              332
Name: count, dtype: int64


In [115]:
region_map = {
    "north america": "North America",
    "n. america": "North America",

    "europe": "Europe",
    "eu": "Europe",

    "asia-pacific": "Asia-Pacific",
    "asia pacific": "Asia-Pacific",
    "apac": "Asia-Pacific",

    "latin america": "Latin America",
    "latam": "Latin America",

    "unknown": "Unknown"
}

df["region"] = (
    df["region"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(region_map)
)

print(df["region"].value_counts(dropna=False))

region
North America    18321
Europe           13222
Asia-Pacific     10524
Latin America     4701
Unknown           2712
Name: count, dtype: int64[pyarrow]


In [116]:
print(df["region"].unique())

<ArrowStringArray>
['North America', 'Latin America', 'Europe', 'Unknown', 'Asia-Pacific']
Length: 5, dtype: string


In [117]:
device_map = {
    "mobile": "Mobile",
    "phone": "Mobile",
    "desktop": "Desktop",
    "tablet": "Tablet",
    "unknown": "Unknown"
}

df["device"] = (
    df["device"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(device_map)
)

print(df["device"].value_counts(dropna=False))

device
Mobile     27399
Desktop    16977
Tablet      3689
Unknown     1415
Name: count, dtype: int64[pyarrow]


In [118]:
print(df.isna().sum())

customer_id                0
date                       0
channel                    0
campaign                 986
age                        0
gender                     0
region                     0
device                     0
is_new_customer            0
prior_purchases            0
pages_viewed            1930
session_duration_sec    2457
email_opened            1982
acquisition_cost          56
converted               1694
revenue                 2426
dtype: int64


In [119]:
df["campaign"] = df["campaign"].fillna("Unknown")

In [120]:
df["pages_viewed"].isna().sum()

np.int64(1930)

In [121]:
df["session_duration_sec"].isna().sum()

np.int64(2457)

In [122]:
print(df["email_opened"].value_counts(dropna=False))

email_opened
0.0    26144
1.0    21354
NaN     1982
Name: count, dtype: int64


In [123]:
df["email_opened"] = df["email_opened"].astype("Int64")

In [124]:
print(df["email_opened"].value_counts(dropna=False))

email_opened
0       26144
1       21354
<NA>     1982
Name: count, dtype: Int64


In [125]:
print("Converted missing + revenue > 0:",
      ((df["converted"].isna()) & (df["revenue"] > 0)).sum())

print("Converted missing + revenue = 0:",
      ((df["converted"].isna()) & (df["revenue"] == 0)).sum())

print("Converted = 1 + revenue missing:",
      ((df["converted"] == 1) & (df["revenue"].isna())).sum())

print("Converted = 0 + revenue missing:",
      ((df["converted"] == 0) & (df["revenue"].isna())).sum())

Converted missing + revenue > 0: 73
Converted missing + revenue = 0: 1585
Converted = 1 + revenue missing: 159
Converted = 0 + revenue missing: 2231


In [126]:
df.loc[
    df["converted"].isna() & (df["revenue"] > 0),
    "converted"
] = 1

In [127]:
print(
    df.loc[
        df["converted"].isna(),
        ["converted", "revenue"]
    ].value_counts(dropna=False)
)

converted  revenue
<NA>       0.0        1585
           <NA>         36
Name: count, dtype: int64


In [128]:
df.loc[
    df["converted"].isna() & (df["revenue"] == 0),
    "converted"
] = 0

df["converted"] = df["converted"].astype("Int64")

In [129]:
print(df["converted"].value_counts(dropna=False))

converted
0       44026
1        5418
<NA>       36
Name: count, dtype: Int64


In [130]:
df.loc[
    (df["age"] < 18) | (df["age"] > 100),
    "age"
] = np.nan

In [131]:
print("Missing age:", df["age"].isna().sum())
print("Age range:", df["age"].min(), "to", df["age"].max())

Missing age: 0
Age range: 18.0 to 70.0


In [132]:
df.loc[
    df["pages_viewed"] < 0,
    "pages_viewed"
] = np.nan

In [133]:
print("Missing pages:", df["pages_viewed"].isna().sum())
print("Minimum pages:", df["pages_viewed"].min())

Missing pages: 2050
Minimum pages: 0.0


In [134]:
print("Missing session duration:",
      df["session_duration_sec"].isna().sum())

print("Minimum session duration:",
      df["session_duration_sec"].min())

print("Maximum session duration:",
      df["session_duration_sec"].max())

Missing session duration: 2457
Minimum session duration: -1094.0
Maximum session duration: 250000.0


In [135]:
df.loc[
    (df["session_duration_sec"] < 0) |
    (df["session_duration_sec"] >= 86400),
    "session_duration_sec"
] = np.nan

In [136]:
print("Missing session duration:",
      df["session_duration_sec"].isna().sum())

print("Minimum:",
      df["session_duration_sec"].min())

print("Maximum:",
      df["session_duration_sec"].max())

Missing session duration: 3176
Minimum: 10.0
Maximum: 1394.0


In [137]:
print("Missing age:", df["age"].isna().sum())
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

Missing age: 0
Minimum age: 18.0
Maximum age: 70.0


In [138]:
df.loc[
    (df["age"] < 18) | (df["age"] > 100),
    "age"
] = np.nan

In [139]:
print("Missing age:", df["age"].isna().sum())
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

Missing age: 0
Minimum age: 18.0
Maximum age: 70.0


In [140]:
print("Missing acquisition cost:",
      df["acquisition_cost"].isna().sum())

print("Minimum acquisition cost:",
      df["acquisition_cost"].min())

print("Maximum acquisition cost:",
      df["acquisition_cost"].max())

Missing acquisition cost: 56
Minimum acquisition cost: 0.0
Maximum acquisition cost: 5000.0


In [141]:
print("=" * 50)
print("NUMERIC DATA VALIDATION")
print("=" * 50)

print("\nAGE")
print("Missing:", df["age"].isna().sum())
print("Min:", df["age"].min())
print("Max:", df["age"].max())

print("\nPAGES VIEWED")
print("Missing:", df["pages_viewed"].isna().sum())
print("Min:", df["pages_viewed"].min())
print("Max:", df["pages_viewed"].max())

print("\nSESSION DURATION")
print("Missing:", df["session_duration_sec"].isna().sum())
print("Min:", df["session_duration_sec"].min())
print("Max:", df["session_duration_sec"].max())

print("\nACQUISITION COST")
print("Missing:", df["acquisition_cost"].isna().sum())
print("Min:", df["acquisition_cost"].min())
print("Max:", df["acquisition_cost"].max())

NUMERIC DATA VALIDATION

AGE
Missing: 0
Min: 18.0
Max: 70.0

PAGES VIEWED
Missing: 2050
Min: 0.0
Max: 1000.0

SESSION DURATION
Missing: 3176
Min: 10.0
Max: 1394.0

ACQUISITION COST
Missing: 56
Min: 0.0
Max: 5000.0


In [142]:
print("=" * 60)
print("FINAL DATA QUALITY CHECK")
print("=" * 60)

print("\n1. SHAPE")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\n2. DUPLICATES")
print("Duplicate rows:", df.duplicated().sum())

print("\n3. MISSING VALUES")
print(df.isna().sum())
print("Total missing cells:", df.isna().sum().sum())

print("\n4. DATA TYPES")
print(df.dtypes)

print("\n5. DATE RANGE")
print("Start:", df["date"].min())
print("End:", df["date"].max())

print("\n6. CATEGORICAL VALUES")
print("\nChannel:")
print(df["channel"].value_counts(dropna=False))

print("\nCampaign:")
print(df["campaign"].value_counts(dropna=False).head(20))

print("\nRegion:")
print(df["region"].value_counts(dropna=False))

print("\nDevice:")
print(df["device"].value_counts(dropna=False))

print("\nGender:")
print(df["gender"].value_counts(dropna=False))

FINAL DATA QUALITY CHECK

1. SHAPE
Rows: 49480
Columns: 16

2. DUPLICATES
Duplicate rows: 2

3. MISSING VALUES
customer_id                0
date                       0
channel                    0
campaign                   0
age                        0
gender                     0
region                     0
device                     0
is_new_customer            0
prior_purchases            0
pages_viewed            2050
session_duration_sec    3176
email_opened            1982
acquisition_cost          56
converted                 36
revenue                 2426
dtype: int64
Total missing cells: 9726

4. DATA TYPES
customer_id                        str
date                    datetime64[ns]
channel                         string
campaign                           str
age                            float64
gender                          string
region                          string
device                          string
is_new_customer                  int64
prior_purchases     

In [143]:
df = df.drop_duplicates().copy()

print("Rows after removing duplicates:", len(df))
print("Duplicate rows:", df.duplicated().sum())

Rows after removing duplicates: 49478
Duplicate rows: 0


In [144]:
print("New customer with prior purchases:",
      ((df["is_new_customer"] == 1) &
       (df["prior_purchases"] > 0)).sum())

print("Existing customer with zero prior purchases:",
      ((df["is_new_customer"] == 0) &
       (df["prior_purchases"] == 0)).sum())

New customer with prior purchases: 991
Existing customer with zero prior purchases: 1934


In [145]:
print("Converted = 0 but revenue > 0:",
      ((df["converted"] == 0) &
       (df["revenue"] > 0)).sum())

print("Converted = 1 but revenue = 0:",
      ((df["converted"] == 1) &
       (df["revenue"] == 0)).sum())

print("Converted = 1 but revenue missing:",
      ((df["converted"] == 1) &
       (df["revenue"].isna())).sum())

print("Converted missing:",
      df["converted"].isna().sum())

Converted = 0 but revenue > 0: 0
Converted = 1 but revenue = 0: 0
Converted = 1 but revenue missing: 159
Converted missing: 36


In [146]:
print("Negative prior purchases:",
      (df["prior_purchases"] < 0).sum())

print("Negative pages viewed:",
      (df["pages_viewed"] < 0).sum())

print("Negative session duration:",
      (df["session_duration_sec"] < 0).sum())

print("Negative acquisition cost:",
      (df["acquisition_cost"] < 0).sum())

print("Negative revenue:",
      (df["revenue"] < 0).sum())

Negative prior purchases: 0
Negative pages viewed: 0
Negative session duration: 0
Negative acquisition cost: 0
Negative revenue: 0


In [147]:
print("New customer with prior purchases:",
      ((df["is_new_customer"] == 1) &
       (df["prior_purchases"] > 0)).sum())

print("Existing customer with zero prior purchases:",
      ((df["is_new_customer"] == 0) &
       (df["prior_purchases"] == 0)).sum())

New customer with prior purchases: 991
Existing customer with zero prior purchases: 1934


In [148]:
df["customer_logic_issue"] = (
    ((df["is_new_customer"] == 1) & (df["prior_purchases"] > 0)) |
    ((df["is_new_customer"] == 0) & (df["prior_purchases"] == 0))
)

print("Customer logic issues:",
      df["customer_logic_issue"].sum())

Customer logic issues: 2925


In [149]:
df = df.drop(columns=["customer_logic_issue"])

In [150]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Normalize customer IDs
df["customer_id"] = (
    df["customer_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Ensure binary fields use nullable integer type
df["email_opened"] = df["email_opened"].astype("Int64")
df["converted"] = df["converted"].astype("Int64")

# Ensure numeric columns use numeric types
numeric_cols = [
    "age",
    "prior_purchases",
    "pages_viewed",
    "session_duration_sec",
    "acquisition_cost",
    "revenue"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Final validation
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())
print("Total missing cells:", df.isna().sum().sum())

Rows: 49478
Columns: 16
Duplicate rows: 0
Total missing cells: 9724


In [151]:
df_clean = df.copy()

In [152]:
df_clean.to_csv("marketing_campaign_cleaned.csv", index=False)

In [153]:
print(df_clean.shape)
print(df_clean.duplicated().sum())

(49478, 16)
0
